In [1]:
import pandas as pd
import numpy as np

import random
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


random.seed(3001)

### Utilities

In [ ]:
def load_titanic():
    # Public CSV
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    y = df["Survived"].astype(int)
    X = df[["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]].copy()
    return X, y

def load_boston():
    # Clean Boston Housing
    url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
    df = pd.read_csv(url)
    y = df["medv"].astype(float)
    X = df.drop(columns=["medv"]).copy()
    return X, y

def sample_hparams():
    dmin, dmax = HPARAM_RANGES["max_depth"]
    smin, smax = HPARAM_RANGES["min_samples_split"]
    lmin, lmax = HPARAM_RANGES["min_samples_leaf"]
    return {
        "max_depth": random.randint(dmin, dmax),
        "min_samples_split": random.randint(smin, smax),
        "min_samples_leaf": random.randint(lmin, lmax)
    }

def make_model(hp):
    if DATASET == "titanic":
        crit = random.choice(CRITERIA_CLASS)
        model = DecisionTreeClassifier(
            criterion=crit,
            max_depth=hp["max_depth"],
            min_samples_split=hp["min_samples_split"],
            min_samples_leaf=hp["min_samples_leaf"],
            random_state=RANDOM_STATE
        )
    else:
        crit = random.choice(CRITERIA_REGR)
        model = DecisionTreeRegressor(
            criterion=crit,
            max_depth=hp["max_depth"],
            min_samples_split=hp["min_samples_split"],
            min_samples_leaf=hp["min_samples_leaf"],
            random_state=RANDOM_STATE
        )
    return model, crit

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    r2_score, mean_absolute_error
)
from sklearn.inspection import permutation_importance
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def sample_hparams():
    return {
        "max_depth":        random.randint(*HPARAM_RANGES["max_depth"]),
        "min_samples_split":random.randint(*HPARAM_RANGES["min_samples_split"]),
        "min_samples_leaf": random.randint(*HPARAM_RANGES["min_samples_leaf"])
    }

def make_model(hp):
    if DATASET == "titanic":
        crit = random.choice(CRITERIA_CLASS)
        model = DecisionTreeClassifier(
            criterion=crit, random_state=RANDOM_STATE, **hp
        )
    else:
        crit = random.choice(CRITERIA_REGR)
        model = DecisionTreeRegressor(
            criterion=crit, random_state=RANDOM_STATE, **hp
        )
    return model, crit

def fit_eval(tag):
    hp = sample_hparams()
    model, crit = make_model(hp)
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)

    # metrics
    y_pred = pipe.predict(X_test)
    if DATASET == "titanic":
        metrics = {
            "Accuracy":  accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall":    recall_score(y_test, y_pred, zero_division=0),
            "F1":        f1_score(y_test, y_pred, zero_division=0),
        }
    else:
        metrics = {
            "R2":  r2_score(y_test, y_pred),
            "MAE": mean_absolute_error(y_test, y_pred),
        }

    # Permutation importance on ORIGINAL features
    orig_feature_names = num_cols + cat_cols
    imp = permutation_importance(pipe, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE)
    imps_perm = (pd.Series(imp.importances_mean, index=orig_feature_names)
                   .sort_values(ascending=False)
                   .head(10))

    return {
        "tag": tag,
        "hparams": hp | {"criterion": crit},
        "pipe": pipe,
        "metrics": metrics,
        "imps_perm": imps_perm,
        # "imps_tree": imps_tree,  # optional
    }

# Tune That Tree!

In [ ]:
# Choose dataset: 'titanic' (classification) or 'boston' (regression)
DATASET = 'titanic'
#DATASET = 'boston'

# Hyperparameter ranges (both models are sampled independently)
HPARAM_RANGES = {
    "max_depth": (3,15 ), #inclusive
    "min_samples_split": (2,30 ),
    "min_samples_leaf": (1,20 )
}

# Criteria for task type
CRITERIA_CLASS = ["gini", "entropy"]
CRITERIA_REGR  = ["squared_error", "absolute_error"]

# Splitting
RANDOM_STATE = 42
TEST_SIZE = 0.25

# Load dataset
if DATASET == "titanic":
    X, y = load_titanic()
else:
    X, y = load_boston()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=(y if DATASET=='titanic' else None))

# Preprocess
cat_cols = X_train.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
num_pre = Pipeline(steps=[("imp", SimpleImputer(strategy="median"))])
cat_pre = Pipeline(steps=[("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer(transformers=[("num", num_pre, num_cols), ("cat", cat_pre, cat_cols)])

In [ ]:
print(f"Dataset: {DATASET} | X shape: {X.shape} | y shape: {y.shape}")
display(X.head(3))

Dataset: titanic | X shape: (891, 7) | y shape: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S


# Compare 2 Trees With Random Parameters

In [ ]:
A = fit_eval("Model A")
B = fit_eval("Model B")

print(" Model A metrics:", A["metrics"])
print(" Model B metrics:", B["metrics"])

 Model A metrics: {'Accuracy': 0.7892376681614349, 'Precision': 0.76, 'Recall': 0.6627906976744186, 'F1': 0.7080745341614907}
 Model B metrics: {'Accuracy': 0.757847533632287, 'Precision': 0.7424242424242424, 'Recall': 0.5697674418604651, 'F1': 0.6447368421052632}


In [ ]:
# Metrics comparison
if DATASET == "titanic":
    mcols = ["Accuracy","Precision","Recall","F1"]
    a_vals = [A["metrics"][m] for m in mcols]
    b_vals = [B["metrics"][m] for m in mcols]
else:
    mcols = ["R2","MAE"]
    a_vals = [A["metrics"][m] for m in mcols]
    b_vals = [B["metrics"][m] for m in mcols]


print(" Model A params:", A["hparams"])
print("\n Model B params:", B["hparams"])

fig1 = go.Figure()
fig1.add_bar(name="Model A", x=mcols, y=a_vals)
fig1.add_bar(name="Model B", x=mcols, y=b_vals)
fig1.update_layout(barmode="group", title=f"{DATASET.title()} — Metrics Comparison for Model A and B")
fig1.show()

 Model A metrics: {'Accuracy': 0.757847533632287, 'Precision': 0.7424242424242424, 'Recall': 0.5697674418604651, 'F1': 0.6447368421052632}
 Model B metrics: {'Accuracy': 0.7668161434977578, 'Precision': 0.6976744186046512, 'Recall': 0.6976744186046512, 'F1': 0.6976744186046512}
 Model A params: {'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 2, 'criterion': 'entropy'}

 Model B params: {'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 7, 'criterion': 'gini'}


In [ ]:
# Top 10 feature importances
impA = A["imps_perm"]
impB = B["imps_perm"]

print(" Model A params:", A["hparams"])
print("\n Model B params:", B["hparams"])

fig2 = make_subplots(rows=1, cols=2, subplot_titles=("Model A — Top Permutation Features",
                                                     "Model B — Top Permutation Features"))
fig2.add_bar(x=impA.values[::-1], y=impA.index[::-1], orientation='h', name="A", row=1, col=1)
fig2.add_bar(x=impB.values[::-1], y=impB.index[::-1], orientation='h', name="B", row=1, col=2)
fig2.update_layout(title=f"{DATASET.title()} — Permutation Importances (Top 10)")
fig2.show()

 Model A params: {'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 2, 'criterion': 'entropy'}

 Model B params: {'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 7, 'criterion': 'gini'}
